In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import polars as pl
from polars import col, lit, when
import re

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from scripts.feature_calculation import build_processed_dataset
from scripts.feature_engineering import compute_global_stats
from scripts.train_eval_model import train_baseline
from scripts.train_eval_model import train_v2

### Creating train dataframe with new features

При считывании создаем колонку, по которой можно отличить train от pretrain и приводим данные к одинаковой схеме

In [3]:
train_1 = pl.scan_parquet('../../data/train_part_1.parquet')
train_1 = train_1.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_1 = pl.scan_parquet('../../data/pretrain_part_1.parquet')
pretrain_1 = pretrain_1.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_2 = pl.scan_parquet('../../data/train_part_2.parquet')
train_2 = train_2.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_2 = pl.scan_parquet('../../data/pretrain_part_2.parquet')
pretrain_2 = pretrain_2.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_3 = pl.scan_parquet('../../data/train_part_3.parquet')
train_3 = train_3.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_3 = pl.scan_parquet('../../data/pretrain_part_3.parquet')
pretrain_3 = pretrain_3.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

pretest = pl.scan_parquet('../../data/pretest.parquet')
pretest = pretest.with_columns(
    pl.lit(0).alias('is_train')
)
test = pl.scan_parquet('../../data/test.parquet')
test = test.with_columns(
    pl.lit(1).alias('is_train')
)

labels = pl.scan_parquet('../../data/train_labels.parquet')

In [4]:
full_train = pl.concat([pretrain_1, pretrain_2, pretrain_3, 
                        train_1, train_2, train_3, pretest, test], how='vertical') 

In [5]:
import shutil

# Compute population-level statistics from the full pre-test history
# (pretrain + train). This covers all data available before the test period,
# giving the most stable estimates for global frequencies and amount
# distributions. The resulting tables are frozen here and injected into
# build_processed_dataset so that Sections F and G use proper training-set
# statistics instead of falling back to per-customer cumulative proxies.
#
# labels_lf is passed so that compute_global_stats can also compute
# Bayesian-smoothed target encodings for event_type_nm, event_desc,
# channel_indicator_type, channel_indicator_sub_type, the
# (event_type_nm, event_desc) pair, and the two within-group channel
# encodings.  Without it those 7 features default to the constant 0.5.
#
# Note: pretest and test are intentionally excluded — they contain transactions
# from the test period and must not influence the population-level statistics
# used to score those same transactions.
history_lf = pl.concat([pretrain_1, pretrain_2, pretrain_3,
                         train_1,    train_2,    train_3])
global_stats = compute_global_stats(history_lf, labels_lf=labels)

# Clear the existing output so that partitions built without global_stats
# do not persist alongside the newly generated ones.
shutil.rmtree('../data_processed/', ignore_errors=True)
shutil.rmtree('../data_splits/', ignore_errors=True)

# build_processed_dataset(full_train, global_stats=global_stats) 
build_processed_dataset(full_train, global_stats=global_stats, labels_lf=labels)

  100,000 customers → 50 partitions × ~2,000 customers each
[██████████████████████████████] 100.0%  part 50/50  (1,707,096 rows)  elapsed 5m00s  ETA 0s                

Done. 86,311,523 total rows written across 50 files in '../data_processed/'.


Посмотрим что получилось

In [6]:
example_data = pl.read_parquet('../data_processed/part_0000.parquet')
example_data.head()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,is_train,hour,day_of_week,day_of_month,week_of_year,hour_of_day,is_weekend,is_night,is_working_hour,minutes_from_midnight,log_amount,amount_abs,amount_round_100,amount_round_1000,…,tx_type_spend_vs_90d,is_new_channel_desc_combo,is_new_channel_type_combo,is_new_subchannel_type_combo,is_new_type_desc_combo,is_new_txtype_channel_combo,amount_zscore_event_desc_global,amount_zscore_event_type_global,amount_zscore_subchannel_global,amount_zscore_pos_global,global_subchannel_freq,amount_zscore_channel_type_subtype_global,global_channel_type_subtype_freq,amount_zscore_evtype_channel_global,global_evtype_channel_freq,amount_zscore_evtype_subchannel_global,global_evtype_subchannel_freq,amount_zscore_evtype_mcc_global,global_evtype_mcc_freq,event_type_nm_target_enc,event_desc_target_enc,channel_type_target_enc,channel_subtype_target_enc,mcc_target_enc,channel_type_subtype_target_enc,evtype_channel_target_enc,evtype_subchannel_target_enc,type_desc_pair_target_enc,channel_type_fraud_rate_within_group,channel_subtype_fraud_rate_within_group,evtype_mcc_target_enc,is_very_high_risk_desc,is_high_risk_desc,is_low_risk_desc,is_high_risk_channel,is_p2p_danger_channel,is_near_certain_fraud_pair
i64,i64,datetime[μs],i32,i32,i32,i32,f32,i32,str,i32,str,str,i32,i64,i32,str,str,str,str,i32,i32,str,i32,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,f32,i8,i8,…,f32,i8,i8,i8,i8,i8,f32,f32,f32,f32,u32,f32,u32,f32,u32,f32,u32,f32,u32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8
123123123123129,123999300382879,2024-10-01 05:29:14,14,75,6,5,56422.0,0,"""4""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,5,2,1,40,5,0,1,0,73,10.940632,56422.0,0,0,…,1.522117,0,0,0,0,0,-0.028531,-0.008474,-0.066327,-0.105602,52449209,-0.040757,27596579,-0.033203,29746895,-0.030952,49445372,-0.01112,26000967,0.649522,0.730438,0.813503,0.702705,0.525717,0.799481,0.831068,0.751336,0.730438,0.801452,0.706026,0.525838,0,0,0,1,0,0
123123123123129,124531875713936,2024-10-01 10:17:22,7,56,4,15,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,10,2,1,40,10,0,0,1,105,0.0,0.0,1,1,…,0.0,0,0,0,0,0,0.0,0.0,-0.012095,0.0,80798395,-0.01192,79820557,0.0,52073384,0.0,52073384,0.0,0,0.581235,0.578522,0.549743,0.550141,0.587769,0.549887,0.582024,0.582024,0.581235,0.573346,0.573346,0.587769,0,0,0,0,0,0
123123123123129,123329285580171,2024-10-01 10:20:03,3,120,6,5,300870.0,0,"""10""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,10,2,1,40,10,0,0,1,108,12.614437,300870.0,0,0,…,1.522031,0,0,0,0,0,-0.434172,-0.077838,0.023197,-0.04722,52449209,0.087627,27596579,-0.419877,1145782,-0.403478,1007013,-0.434185,1861304,0.588468,0.596224,0.813503,0.702705,0.661786,0.799481,0.612254,0.594418,0.596224,0.801452,0.706026,0.596224,0,0,0,1,0,0
123123123123129,124334305430665,2024-10-02 07:48:09,14,75,6,5,298458.0,0,"""1""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,7,3,2,40,7,0,0,0,-44,12.606388,298458.0,0,0,…,1.522205,0,0,0,0,0,0.100807,-0.007121,0.022314,-0.047796,52449209,0.086361,27596579,0.082399,29746895,0.092165,49445372,0.023025,2218436,0.649522,0.730438,0.813503,0.702705,0.319538,0.799481,0.831068,0.751336,0.730438,0.801452,0.706026,0.319538,0,0,0,1,0,0
123123123123129,126215501146513,2024-10-02 11:20:40,14,75,6,5,59944.0,0,"""15""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,11,3,2,40,11,0,0,1,-88,11.001183,59944.0,0,0,…,1.521751,0,0,0,0,0,-0.026649,-0.008454,-0.065037,-0.104761,52449209,-0.038908,27596579,-0.031521,29746895,-0.029161,49445372,-0.007464,5293421,0.649522,0.730438,0.813503,0.702705,0.515121,0.799481,0.831068,0.751336,0.730438,0.801452,0.706026,0.5

### Training baseline on engineered features

In [7]:
train_baseline()

  Found 20 existing model file(s) - archiving as _ver_5 ...
    model_card_catboost.cbm  ->  model_card_catboost_ver_5.cbm
    model_card_s0.txt  ->  model_card_s0_ver_5.txt
    model_card_s1.txt  ->  model_card_s1_ver_5.txt
    model_card_s2.txt  ->  model_card_s2_ver_5.txt
    model_np_other_catboost.cbm  ->  model_np_other_catboost_ver_5.cbm
    model_np_other_s0.txt  ->  model_np_other_s0_ver_5.txt
    model_np_other_s1.txt  ->  model_np_other_s1_ver_5.txt
    model_np_other_s2.txt  ->  model_np_other_s2_ver_5.txt
    model_np_other_s3.txt  ->  model_np_other_s3_ver_5.txt
    model_np_other_s4.txt  ->  model_np_other_s4_ver_5.txt
    model_np_type7_catboost.cbm  ->  model_np_type7_catboost_ver_5.cbm
    model_np_type7_s0.txt  ->  model_np_type7_s0_ver_5.txt
    model_np_type7_s1.txt  ->  model_np_type7_s1_ver_5.txt
    model_np_type7_s2.txt  ->  model_np_type7_s2_ver_5.txt
    model_p2p_catboost.cbm  ->  model_p2p_catboost_ver_5.cbm
    model_p2p_s0.txt  ->  model_p2p_s0_ver_5.txt


In [3]:
train_v2()

Found 50 parquet partitions in '../data_processed'

Device                : CPU
LGBM ensemble seeds   : [42]
Yellow weight         : 2.0
Hier full features    : True
Blend in logit space  : True
Blend on all rows     : True
Train recent LGBM     : True
Green weights         : old=3.0  recent=1.5

  Step 1 — Build memmap splits

  Cache hit — reusing memmap files from '../data_splits'
    X_train       : 60,916,817 × 462
    X_val         : 24,761,023 × 462
    Labeled in val: 26,865 / 24,761,023
    Features      : 462  (event_type_nm … is_near_certain_fraud_pair)


  Feature columns : 462
  X_train shape   : (60916817, 462)
  X_val shape     : (24761023, 462)

  Hier features (full) : 464 (451 num + 13 cat)
  Green ratio          : 0.05

  Step 2 — Pre-load val from parquet

Loading labels …
  87,514 labelled rows  |  51,438 positives  (58.777%)

  Val loaded: 274,736 rows  (26,865 labeled [13,686 pos], 247,871 unlabeled)

  Step 3 — Per-group LightGBM


──────────────────────────────

C:\Users\YOGA\Documents\data_fusion_2026\data_fusion_2026\scripts\training\pipeline_v2.py:1197: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hier_full_df["__susp_target"] = (
C:\Users\YOGA\Documents\data_fusion_2026\data_fusion_2026\scripts\training\pipeline_v2.py:1207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hier_full_df["__susp_weight"] = w_susp


0:	total: 7.04s	remaining: 6h 9m 20s
100:	total: 9m 21s	remaining: 4h 42m 18s
200:	total: 19m 2s	remaining: 4h 39m 28s
300:	total: 28m 23s	remaining: 4h 28m 42s
400:	total: 38m 19s	remaining: 4h 22m 46s
500:	total: 47m 51s	remaining: 4h 13m 1s
600:	total: 57m 12s	remaining: 4h 2m 39s
700:	total: 1h 6m 26s	remaining: 3h 52m 6s
800:	total: 1h 15m 36s	remaining: 3h 41m 42s
900:	total: 1h 24m 50s	remaining: 3h 31m 46s
1000:	total: 1h 33m 51s	remaining: 3h 21m 30s
1100:	total: 1h 42m 58s	remaining: 3h 11m 37s
1200:	total: 1h 52m 4s	remaining: 3h 1m 53s
1300:	total: 2h 1m 4s	remaining: 2h 52m 4s
1400:	total: 2h 9m 54s	remaining: 2h 42m 10s
1500:	total: 2h 18m 56s	remaining: 2h 32m 38s
1600:	total: 2h 27m 43s	remaining: 2h 22m 55s
1700:	total: 2h 37m 9s	remaining: 2h 13m 52s
1800:	total: 2h 45m 53s	remaining: 2h 4m 15s
1900:	total: 2h 54m 47s	remaining: 1h 54m 50s
2000:	total: 3h 3m 50s	remaining: 1h 45m 33s
2100:	total: 3h 13m 3s	remaining: 1h 36m 23s
2200:	total: 3h 22m 19s	remaining: 1h 27

C:\Users\YOGA\Documents\data_fusion_2026\data_fusion_2026\scripts\training\pipeline_v2.py:1272: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hier_full_df["__main_target"] = (
C:\Users\YOGA\Documents\data_fusion_2026\data_fusion_2026\scripts\training\pipeline_v2.py:1282: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hier_full_df["__main_weight"] = w_main_arr


0:	learn: 0.6252739	total: 8.05s	remaining: 6h 29m 2s
100:	learn: 0.8489632	total: 10m 31s	remaining: 4h 51m 31s
200:	learn: 0.8849207	total: 21m 24s	remaining: 4h 47m 16s
300:	learn: 0.9063012	total: 33m 11s	remaining: 4h 46m 26s
400:	learn: 0.9198404	total: 43m 52s	remaining: 4h 33m 19s
500:	learn: 0.9276967	total: 53m 38s	remaining: 4h 16m 44s
600:	learn: 0.9312076	total: 1h 3m 23s	remaining: 4h 2m 23s
700:	learn: 0.9351057	total: 1h 13m 10s	remaining: 3h 49m 26s
800:	learn: 0.9381884	total: 1h 22m 44s	remaining: 3h 36m 43s
900:	learn: 0.9409257	total: 1h 32m 15s	remaining: 3h 24m 35s
1000:	learn: 0.9440636	total: 1h 42m 3s	remaining: 3h 13m 30s
1100:	learn: 0.9460219	total: 1h 51m 42s	remaining: 3h 2m 25s
1200:	learn: 0.9482142	total: 2h 2m 20s	remaining: 2h 52m 58s
1300:	learn: 0.9500717	total: 2h 12m 52s	remaining: 2h 43m 12s
1400:	learn: 0.9521083	total: 2h 23m 5s	remaining: 2h 32m 59s
1500:	learn: 0.9534036	total: 2h 33m 24s	remaining: 2h 22m 52s
1600:	learn: 0.9550135	total: 2

### Resume training code, skip for now

In [4]:
from scripts.train_eval_model import resume_from_step10b

In [5]:
resume_from_step10b(
      susp_best_iter=3000,
      rgs_best_iter=704,
      recent_best_iter=1277,
      w_lgbm=0.10,
      w_hier=0.80,
      w_recent=0.10,
)

Feature columns   : 462
Blend weights     : w_lgbm=0.10  w_hier=0.80  w_recent=0.10
Best iters        : susp=3000  rgs=704  recent=1277
LGBM models dir   : models

  Reloading saved LGBM models (Step 10a output)

  Loaded models\model_np_type7_s0.txt
  Loaded models\model_np_type7_s1.txt
  Loaded models\model_np_type7_s2.txt
  Loaded models\model_np_type7_s3.txt
  Loaded models\model_np_type7_s4.txt
  Loaded models\model_np_other_s0.txt
  Loaded models\model_np_other_s1.txt
  Loaded models\model_np_other_s2.txt
  Loaded models\model_np_other_s3.txt
  Loaded models\model_np_other_s4.txt
  Loaded models\model_card_s0.txt
  Loaded models\model_card_s1.txt
  Loaded models\model_card_s2.txt
  Loaded models\model_card_s3.txt
  Loaded models\model_card_s4.txt
  Loaded models\model_p2p_s0.txt
  Loaded models\model_p2p_s1.txt
  Loaded models\model_p2p_s2.txt
  Loaded models\model_p2p_s3.txt
  Loaded models\model_p2p_s4.txt

Loading labels …
  87,514 labelled rows  |  51,438 positives  (58.777%)

### Submission

In [4]:
sub = pd.read_csv('submission.csv')
sub.shape

(633683, 2)

In [5]:
sub.head()

,event_id,predict
0,123707242230467,0.000063
1,123234793229123,0.000209
2,125837545055907,0.000276
3,126456020999239,0.000192
4,125090221587312,0.013760


### Feature importance analysis

(Устарело)

In [11]:
import lightgbm as lgb

bst = lgb.Booster(model_file='baseline_lgbm.txt')

In [22]:
model = lgb.Booster(model_file='baseline_lgbm.txt')

In [23]:
model.feature_name()

['event_type_nm',
 'event_desc',
 'channel_indicator_type',
 'channel_indicator_sub_type',
 'operaton_amt',
 'currency_iso_cd',
 'pos_cd',
 'timezone',
 'session_id',
 'operating_system_type',
 'phone_voip_call_state',
 'web_rdp_connection',
 'hour',
 'day_of_week',
 'day_of_month',
 'week_of_year',
 'hour_of_day',
 'is_weekend',
 'is_night',
 'is_working_hour',
 'minutes_from_midnight',
 'log_amount',
 'amount_abs',
 'amount_round_100',
 'amount_round_1000',
 'amount_is_integer',
 'amount_missing_flag',
 'amount_currency_mismatch_flag',
 'amount_usd_normalized',
 'is_month_start',
 'is_month_end',
 'is_payday',
 'is_holiday',
 'tx_count_lifetime',
 'time_since_last_tx_minutes',
 'mcc_freq_user_cum',
 'pos_freq_user_cum',
 'event_desc_user_freq',
 'event_type_user_freq',
 'mcc_is_new_for_user',
 'pos_cd_is_new',
 'mcc_transaction_share_user',
 'pos_cd_transaction_share_user',
 'merchant_switch_flag',
 'time_since_last_3_tx_mean',
 'mcc_frequency_user',
 'event_desc_is_new_for_user',
 '

In [24]:
importances = importances = model.feature_importance(importance_type='gain')
feature_importance_df = pd.DataFrame({
    'Feature': model.feature_name(),
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [25]:
feature_importance_df.head(30)

,Feature,Importance
44,time_since_last_3_tx_mean,137966.299146
213,global_mcc_freq,90486.215285
37,event_desc_user_freq,90138.605416
190,spend_in_channel_lifetime,61330.408890
220,global_event_desc_freq,59363.082040
4,operaton_amt,58881.362158
237,amount_zscore_given_device,58240.613001
225,time_gap_mean_30d,54442.208632
48,event_desc_share_user,49832.267904
239,amount_zscore_channel,45464.942554


In [26]:
feature_importance_df.to_csv('feature_importances.csv', index=False)